<a href="https://colab.research.google.com/github/beatrisAS/Desafio1_API_Streaming_Filmes_Python/blob/main/Desafio2_Estatistica_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Célula 01 · Importações

**O que esta célula faz:** importa as bibliotecas que o notebook usa (arquivos, Google Colab, pandas e gráficos) e define configurações gerais, que valem para todos os gráficos.
**Não tem lacunas:** copie e execute.

In [ ]:
# ---------- Bibliotecas: ferramentas prontas que o Python usa ----------
import warnings                    # esconde avisos técnicos para não distrair
from pathlib import Path           # manipula caminhos de arquivos e pastas
import shutil                      # manipula arquivos e pastas (ex.: mover, copiar, apagar)

# Estas linhas servem para trabalhar no Google Colab (conectar ao Drive e enviar arquivos)
from google.colab import drive     # conecta o Google Drive ao Colab (para ler e salvar arquivos)
from google.colab import files     # (files) envia arquivos do seu computador para o Colab

# Bibliotecas de análise de dados e de gráficos
import pandas as pd                # tabelas de dados (DataFrame)
import seaborn as sns              # gráficos bonitos e simples
import matplotlib.pyplot as plt    # ajustes nos gráficos (título, linhas...)

# ---------- Configurações gerais (valem para todos os gráficos abaixo) ----------
warnings.filterwarnings("ignore")           # esconde avisos técnicos para não distrair
sns.set_theme(style="whitegrid")            # deixa todos os gráficos com um visual limpo
plt.rcParams["figure.figsize"] = (8, 6)    # tamanho padrão dos gráficos (largura, altura)

### Célula 02 · Carregar os arquivos CSV

**O que esta célula faz:** conecta o Google Drive, guarda os CSV na pasta `Colab Notebooks/desafios/dados` e lê os dois arquivos em DataFrames: `df_dimensao` e `df_fato`. Os nomes das tabelas são os mesmos dos arquivos, sem o `.csv`.

*  Na primeira execução, o Colab pede autorização para acessar o Drive.
* Se algum arquivo ainda não estiver na pasta, abre a janela de seleção: escolha os **2 CSV de uma vez**. O código identifica cada um pelo nome (o nome do arquivo precisa conter o nome da tabela) e salva já com o nome certo.
*  Nas próximas execuções, lê direto do Drive, sem pedir envio.



In [ ]:
# Nomes das tabelas do seu tema (veja a tabela acima), iguais aos nomes dos arquivos CSV.
NOME_TABELA_DIMENSAO = "filmes"   # ex.: "livros"
NOME_TABELA_FATO = "visualizacoes"           # ex.: "emprestimos"

# ---------- 1) Conectando ao Google Drive ----------
drive.mount("/content/drive")      # vai pedir autorização na primeira vez

# ---------- 2) Dizendo onde os arquivos estão guardados ----------
pasta_dados = Path("/content/drive/MyDrive/Colab Notebooks/desafios/dados")   # pasta onde a base fica guardada no Drive
pasta_dados.mkdir(parents=True, exist_ok=True)                                # cria a pasta "dados" se faltar

# O "/" junta a pasta com o nome do arquivo (o nome vem da variável de cada tabela).
caminho_dimensao = pasta_dados / f"{NOME_TABELA_DIMENSAO}.csv"
caminho_fato = pasta_dados / f"{NOME_TABELA_FATO}.csv"

# ---------- 3) Se faltar algum arquivo, envia os dois de uma vez e o código separa ----------
if not caminho_dimensao.exists() or not caminho_fato.exists():
    print(f"Selecione os arquivos CSV ({NOME_TABELA_DIMENSAO} e {NOME_TABELA_FATO}). Pode escolher os dois de uma vez:")
    enviados = files.upload()                                  # abre o seletor; aceita vários arquivos

    for nome_enviado in enviados:                              # percorre cada arquivo enviado
        if NOME_TABELA_DIMENSAO in nome_enviado.lower():       # o nome contém o da dimensão? salva como dimensão
            shutil.move(nome_enviado, caminho_dimensao)        # salva já com o nome da tabela
            print(f"{nome_enviado} -> {caminho_dimensao.name}")
        elif NOME_TABELA_FATO in nome_enviado.lower():         # o nome contém o da fato? salva como fato
            shutil.move(nome_enviado, caminho_fato)
            print(f"{nome_enviado} -> {caminho_fato.name}")
        else:                                                  # nome que não combina com nenhuma tabela
            print(f"Ignorado: {nome_enviado} (o nome não contém '{NOME_TABELA_DIMENSAO}' nem '{NOME_TABELA_FATO}')")

# Confere se os dois arquivos estão no Drive antes de ler.
for caminho in [caminho_dimensao, caminho_fato]:
    if not caminho.exists():
        raise FileNotFoundError(f"Falta o arquivo {caminho.name}. Execute a célula de novo e envie-o.")

# ---------- 4) Lendo os dois arquivos ----------
df_dimensao = pd.read_csv(caminho_dimensao)
df_fato = pd.read_csv(caminho_fato)

print(NOME_TABELA_DIMENSAO, "->", len(df_dimensao), "linhas")
print(NOME_TABELA_FATO, "->", len(df_fato), "linhas")

### Célula 03 · Conhecendo a tabela dimensão

**O que esta célula faz:** mostra quantas linhas e colunas tem a tabela dimensão e as 5 primeiras linhas. Olhe a chave primária e as colunas de cadastro. **Não tem lacunas.**

In [ ]:
# Quantas linhas (registros do cadastro) e colunas (variáveis) tem a tabela dimensão?
# .shape devolve dois números: (quantidade de linhas, quantidade de colunas)
linhas, colunas = df_dimensao.shape
print(f"A tabela dimensão tem {linhas} linhas e {colunas} colunas (variáveis)")
display(df_dimensao.head())

Célula 04 · Conhecendo a tabela fato
O que esta célula faz: o mesmo para a tabela fato. Olhe a chave estrangeira, as datas e as colunas numéricas. Não tem lacunas.

In [ ]:
# Quantas linhas (eventos) e colunas (variáveis) tem a tabela fato?
# .shape devolve dois números: (quantidade de linhas, quantidade de colunas)
linhas, colunas = df_fato.shape
print(f"A tabela fato tem {linhas} linhas e {colunas} colunas (variáveis)")
display(df_fato.head())

In [ ]:
# 1) Tipo de cada coluna e quantidade de valores preenchidos. Compare com o esquema do plano do tema.
df_fato.info()

**Célula 06 · Valores ausentes**  
Conta campos vazios antes dos cálculos.

In [ ]:
# 2) Valores ausentes por coluna (tabela fato).
print("\nValores ausentes por coluna:")
print(df_fato.isna().sum())

**Célula 07 · Duplicatas**  
Verifica se há registros totalmente repetidos.

In [ ]:
# 3) Linhas duplicadas (tabela fato).
print("\nLinhas duplicadas:", df_fato.duplicated().sum())

**Célula 08 · Chave estrangeira**  
Confere se todo filme_id da fato existe na dimensão.

In [ ]:
# 4) Chave estrangeira: todos os registros da tabela fato devem apontar para um ID que existe na tabela dimensão.
COLUNA_FK = "filme_id"        # ex.: "livro_id" (coluna da fato que referencia a dimensão)
COLUNA_ID = "filme_id"   # ex.: "livro_id" (coluna de ID da dimensão)
print("\nChaves estrangeiras que batem:", df_fato[COLUNA_FK].isin(df_dimensao[COLUNA_ID]).all())

**Leitura da conferência · resultados**

A tabela fato `visualizacoes` tem 55.000 registros e 7 colunas. Não foram encontradas linhas completamente duplicadas. Todos os valores de `filme_id` da tabela fato correspondem a um filme existente na tabela dimensão. A coluna `tempo_assistido_min` não tem valores ausentes; nenhuma das demais colunas verificadas da fato tem valores ausentes. Os tipos identificados são inteiros para IDs e medidas, texto para data/hora e dispositivo, e booleano para `concluido`, coerentes com o uso esperado.

**Célula 09 · Coluna de análise**  
Seleciona tempo_assistido_min e confere valores ausentes.

In [ ]:
# Coluna numérica escolhida no Passo 1 (nome EXATO, como está no esquema do tema).
COLUNA_NUMERICA = "tempo_assistido_min"

# A coluna escolhida, como uma série (uma coluna do DataFrame).
serie_numerica = df_fato[COLUNA_NUMERICA]
print("Valores preenchidos:", serie_numerica.count(), "| ausentes:", serie_numerica.isna().sum())

**Célula 10 · Medidas estatísticas**  
Calcula tendência central e dispersão do tempo assistido.

In [ ]:
# Medidas de Tendência central e dispersão.
medidas = pd.Series({
    "Média": serie_numerica.mean(),
    "Mediana": serie_numerica.median(),
    "Moda": serie_numerica.mode()[0],            # se houver empate, a primeira será selecionada
    "Desvio padrão": serie_numerica.std(),
    "Variância": serie_numerica.var(),
    "Mínimo": serie_numerica.min(),
    "Máximo": serie_numerica.max(),
    "Amplitude": serie_numerica.max() - serie_numerica.min(),
}).round(2)
display(medidas)

# Média muito diferente da mediana é sinal de valores extremos ou de dados assimétricos.
print("Média - mediana:", round(medidas["Média"] - medidas["Mediana"], 2))

**Insights · média, mediana e dispersão**

- **Média:** O tempo assistido médio é de **90,93 minutos por registro de visualização**.
- **Mediana:** A mediana é **88 minutos**: metade dos registros tem tempo assistido de até esse valor e metade tem tempo igual ou superior. Ela fica 2,93 minutos abaixo da média.
- **Desvio padrão:** O desvio padrão é **35,26 minutos**, mostrando variação dos tempos em torno da média.
- **Decisão apoiada:** Usar a mediana como referência do tempo típico e comparar grupos para verificar se alguma diferença merece investigação.